## Preprocessing and Feature Engineering Notebook 2:
upgrade primary merged data set with spatial and temporal lags as suggested by 
instructor, to improve the model's historical and regional context,create updated test/train sets in accord with Brendans previous notebook.


### Capstone Project

#### Team Rho

##### *Brendan Collari and Casey Brookshier*

In [1]:
# Preprocessing and Feature Engineering Notebook
# Update with Temporal + Spatial Feature Engineering
# imports and relative paths


import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)



PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "Engineered_Data_v2"

DATA = DATA_DIR / "team_rho_state_year_prediction_dataset_ENGINEERED_v2.csv"

df = pd.read_csv(DATA)

print(df.shape)
df.head()

(452, 14)


,State,Year,state_population,admission_rate_per_100k,suicidal thoughts_lag1,suicide hotline_lag1,self harm_lag1,mental health crisis_lag1,suicide prevention_lag1,crisis hotline_lag1,psychiatric hospital_lag1,depression help_lag1,admission_rate_lag2,neighbor_admission_rate_lag1
0,Alaska,2015,738430.0,297.929391,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,442.970047
1,Alaska,2016,742575.0,289.533044,0.000000,0.000000,16.000000,0.0,9.250000,0.0,0.000000,0.000000,NaN,432.923416
2,Alaska,2017,740983.0,303.650691,0.000000,0.000000,6.333333,0.0,12.750000,0.0,0.000000,13.250000,297.929391,394.821026
3,Alaska,2018,736624.0,319.023002,7.666667,24.333333,7.500000,0.0,3.666667,0.0,0.000000,6.416667,289.533044,407.012291
4,Alaska,2019,733603.0,333.968100,0.000000,22.000000,0.000000,0.0,18.333333,0.0,8.333333,5.583333,303.650691,408.686371


In [2]:
# remove leakage variable to prevent target leakage

df = df.drop(
    columns=["mental_health_admissions"],
    errors="ignore"
)

print(df.shape)
print(df.columns.tolist())

# temporal feature add
# add 2-year admission rate lag

df = df.sort_values(["State", "Year"])

df["admission_rate_lag2"] = (
    df.groupby("State")["admission_rate_per_100k"]
      .shift(2)
)

# spatial feature engineering
# neighbor-state lagged admission rate

neighbors = {

    "Alabama": ["Florida", "Georgia", "Mississippi", "Tennessee"],
    "Alaska": [],
    "Arizona": ["California", "Nevada", "Utah", "New Mexico"],
    "Arkansas": ["Missouri", "Tennessee", "Mississippi", "Louisiana", "Texas", "Oklahoma"],
    "California": ["Oregon", "Nevada", "Arizona"],
    "Colorado": ["Wyoming", "Nebraska", "Kansas", "Oklahoma", "New Mexico", "Utah", "Arizona"],
    "Connecticut": ["Rhode Island", "Massachusetts", "New York"],
    "Delaware": ["Maryland", "Pennsylvania", "New Jersey"],
    "Florida": ["Alabama", "Georgia"],
    "Georgia": ["Florida", "Alabama", "Tennessee", "North Carolina", "South Carolina"],
    "Hawaii": [],
    "Idaho": ["Washington", "Oregon", "Nevada", "Utah", "Wyoming", "Montana"],
    "Illinois": ["Wisconsin", "Iowa", "Missouri", "Kentucky", "Indiana", "Michigan"],
    "Indiana": ["Illinois", "Michigan", "Ohio", "Kentucky"],
    "Iowa": ["Minnesota", "Wisconsin", "Illinois", "Missouri", "Nebraska", "South Dakota"],
    "Kansas": ["Nebraska", "Missouri", "Oklahoma", "Colorado"],
    "Kentucky": ["Illinois", "Indiana", "Ohio", "West Virginia", "Virginia", "Tennessee", "Missouri"],
    "Louisiana": ["Texas", "Arkansas", "Mississippi"],
    "Maine": ["New Hampshire"],
    "Maryland": ["Delaware", "Pennsylvania", "Virginia", "West Virginia"],
    "Massachusetts": ["Rhode Island", "Connecticut", "New York", "Vermont", "New Hampshire"],
    "Michigan": ["Wisconsin", "Illinois", "Indiana", "Ohio"],
    "Minnesota": ["North Dakota", "South Dakota", "Iowa", "Wisconsin"],
    "Mississippi": ["Louisiana", "Arkansas", "Tennessee", "Alabama"],
    "Missouri": ["Iowa", "Illinois", "Kentucky", "Tennessee", "Arkansas", "Oklahoma", "Kansas", "Nebraska"],
    "Montana": ["Idaho", "Wyoming", "South Dakota", "North Dakota"],
    "Nebraska": ["South Dakota", "Iowa", "Missouri", "Kansas", "Colorado", "Wyoming"],
    "Nevada": ["Oregon", "Idaho", "Utah", "Arizona", "California"],
    "New Hampshire": ["Maine", "Vermont", "Massachusetts"],
    "New Jersey": ["New York", "Pennsylvania", "Delaware"],
    "New Mexico": ["Arizona", "Utah", "Colorado", "Oklahoma", "Texas"],
    "New York": ["Pennsylvania", "New Jersey", "Connecticut", "Massachusetts", "Vermont"],
    "North Carolina": ["Virginia", "Tennessee", "Georgia", "South Carolina"],
    "North Dakota": ["Montana", "South Dakota", "Minnesota"],
    "Ohio": ["Michigan", "Indiana", "Kentucky", "West Virginia", "Pennsylvania"],
    "Oklahoma": ["Colorado", "Kansas", "Missouri", "Arkansas", "Texas", "New Mexico"],
    "Oregon": ["Washington", "Idaho", "Nevada", "California"],
    "Pennsylvania": ["New York", "New Jersey", "Delaware", "Maryland", "West Virginia", "Ohio"],
    "Rhode Island": ["Connecticut", "Massachusetts"],
    "South Carolina": ["North Carolina", "Georgia"],
    "South Dakota": ["North Dakota", "Minnesota", "Iowa", "Nebraska", "Wyoming", "Montana"],
    "Tennessee": ["Kentucky", "Virginia", "North Carolina", "Georgia", "Alabama", "Mississippi", "Arkansas", "Missouri"],
    "Texas": ["New Mexico", "Oklahoma", "Arkansas", "Louisiana"],
    "Utah": ["Idaho", "Wyoming", "Colorado", "New Mexico", "Arizona", "Nevada"],
    "Vermont": ["New York", "New Hampshire", "Massachusetts"],
    "Virginia": ["Maryland", "West Virginia", "Kentucky", "Tennessee", "North Carolina"],
    "Washington": ["Oregon", "Idaho"],
    "West Virginia": ["Ohio", "Pennsylvania", "Maryland", "Virginia", "Kentucky"],
    "Wisconsin": ["Michigan", "Minnesota", "Iowa", "Illinois"],
    "Wyoming": ["Montana", "South Dakota", "Nebraska", "Colorado", "Utah", "Idaho"]

}

# create previous-year neighbor lookup
# create small DF containing only state/year/admission rate

neighbor_history = (
    df[["State", "Year", "admission_rate_per_100k"]]
    .copy()
)


# shift year fwd by 1

neighbor_history["Year"] = neighbor_history["Year"] + 1

(452, 14)
['State', 'Year', 'state_population', 'admission_rate_per_100k', 'suicidal thoughts_lag1', 'suicide hotline_lag1', 'self harm_lag1', 'mental health crisis_lag1', 'suicide prevention_lag1', 'crisis hotline_lag1', 'psychiatric hospital_lag1', 'depression help_lag1', 'admission_rate_lag2', 'neighbor_admission_rate_lag1']


In [3]:
# define function to calculate neighbor influence

def get_neighbor_lag(row):

    # look up list of neighbor states
    neigh = neighbors.get(row["State"], [])

    # handle states w/o neighbors
    if len(neigh) == 0:
        return np.nan

    # find previous year neighbor adm. rates
    values = neighbor_history.loc[
        (neighbor_history["State"].isin(neigh)) &
        (neighbor_history["Year"] == row["Year"]),
        "admission_rate_per_100k"
    ]

    return values.mean()


df["neighbor_admission_rate_lag1"] = df.apply(
    get_neighbor_lag,
    axis=1
)

In [4]:
# imputation: fill missing values using the PRIOR year's national average
# (uses neighbor_history, which is already shifted +1 year, so this avoids
# leaking the current year's target value into the predictor)

national_lag_avg = neighbor_history.groupby("Year")["admission_rate_per_100k"].mean()

df["neighbor_admission_rate_lag1"] = (
    df["neighbor_admission_rate_lag1"]
    .fillna(df["Year"].map(national_lag_avg))
)

In [5]:
# save engineered merged dataset

ENGINEERED = (
    DATA_DIR /
    "team_rho_state_year_prediction_dataset_ENGINEERED_v2.csv"
)

df.to_csv(
    ENGINEERED,
    index=False
)

print("Saved engineered dataset:", ENGINEERED)
print(df.shape)
df.head()

Saved engineered dataset: /Users/caseybrookshier/Team-Rho-Capstone-Project/Data/processed/Engineered_Data_v2/team_rho_state_year_prediction_dataset_ENGINEERED_v2.csv
(452, 14)


,State,Year,state_population,admission_rate_per_100k,suicidal thoughts_lag1,suicide hotline_lag1,self harm_lag1,mental health crisis_lag1,suicide prevention_lag1,crisis hotline_lag1,psychiatric hospital_lag1,depression help_lag1,admission_rate_lag2,neighbor_admission_rate_lag1
0,Alaska,2015,738430.0,297.929391,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,442.970047
1,Alaska,2016,742575.0,289.533044,0.000000,0.000000,16.000000,0.0,9.250000,0.0,0.000000,0.000000,NaN,432.923416
2,Alaska,2017,740983.0,303.650691,0.000000,0.000000,6.333333,0.0,12.750000,0.0,0.000000,13.250000,297.929391,394.821026
3,Alaska,2018,736624.0,319.023002,7.666667,24.333333,7.500000,0.0,3.666667,0.0,0.000000,6.416667,289.533044,407.012291
4,Alaska,2019,733603.0,333.968100,0.000000,22.000000,0.000000,0.0,18.333333,0.0,8.333333,5.583333,303.650691,408.686371


In [6]:
# Train/Test split to prevent temporal leakage

train_years = [2014, 2015, 2016, 2017, 2018, 2019, 2020]
test_years = [2021, 2022, 2023]


train_df = df[df["Year"].isin(train_years)].copy()
test_df = df[df["Year"].isin(test_years)].copy()


print(
    f"Train shape: {train_df.shape}, Test shape: {test_df.shape}"
)

Train shape: (319, 14), Test shape: (133, 14)


In [7]:
# remove missing temporal lag rows
# define required lag variables, drop incomplete rows

required_cols = [
    "suicidal thoughts_lag1",
    "admission_rate_lag2"
]


print("Before:", train_df.shape)

train_df = train_df.dropna(
    subset=required_cols
)

test_df = test_df.dropna(
    subset=["admission_rate_lag2"]
)

print("Train after:", train_df.shape)
print("Test after:", test_df.shape)

Before: (319, 14)
Train after: (227, 14)
Test after: (133, 14)


In [8]:
# PCA

search_term_cols = [
    "suicidal thoughts_lag1",
    "suicide hotline_lag1",
    "self harm_lag1",
    "mental health crisis_lag1",
    "suicide prevention_lag1",
    "crisis hotline_lag1",
    "psychiatric hospital_lag1",
    "depression help_lag1"
]

In [9]:
# standardize using training data only

train_values = train_df[search_term_cols].values
test_values = test_df[search_term_cols].values

train_mean = train_values.mean(axis=0)
train_std = train_values.std(axis=0)

train_scaled = (train_values - train_mean) / train_std
test_scaled = (test_values - train_mean) / train_std


print(train_scaled.shape)
print(test_scaled.shape)

(227, 8)
(133, 8)


In [10]:
# PCA using SVD (single value decomposition)

U, S, Vt = np.linalg.svd(
    train_scaled,
    full_matrices=False
)


# explained variance ratio

explained_variance = (S ** 2) / (len(train_scaled) - 1)
explained_ratio = explained_variance / explained_variance.sum()


cumulative = explained_ratio.cumsum()

for i, (ev, cv) in enumerate(
    zip(explained_ratio, cumulative),
    start=1
):
    print(
        f"PC{i}: {ev:.3f} explained | {cv:.3f} cumulative"
    )

PC1: 0.695 explained | 0.695 cumulative
PC2: 0.114 explained | 0.808 cumulative
PC3: 0.093 explained | 0.901 cumulative
PC4: 0.027 explained | 0.928 cumulative
PC5: 0.024 explained | 0.952 cumulative
PC6: 0.021 explained | 0.973 cumulative
PC7: 0.017 explained | 0.990 cumulative
PC8: 0.010 explained | 1.000 cumulative


In [11]:
# keep 3 PCs

n_components = 3

components = Vt[:n_components].T


train_pca_final = train_scaled @ components
test_pca_final = test_scaled @ components


pca_cols = [
    f"search_PC{i+1}"
    for i in range(n_components)
]


train_pca_df = pd.DataFrame(
    train_pca_final,
    columns=pca_cols
)

test_pca_df = pd.DataFrame(
    test_pca_final,
    columns=pca_cols
)

In [12]:
# assemble final train/test datasets

keep_cols = [
    "State",
    "Year",
    "state_population",
    "admission_rate_per_100k",
    "admission_rate_lag2",
    "neighbor_admission_rate_lag1"
]


train_final = pd.concat(
    [
        train_df[keep_cols].reset_index(drop=True),
        train_pca_df.reset_index(drop=True)
    ],
    axis=1
)


test_final = pd.concat(
    [
        test_df[keep_cols].reset_index(drop=True),
        test_pca_df.reset_index(drop=True)
    ],
    axis=1
)


print(train_final.shape)
print(test_final.shape)

(227, 9)
(133, 9)


In [13]:
# log transform population to normalize state pop differences

train_final["log_state_population"] = np.log(
    train_final["state_population"]
)

test_final["log_state_population"] = np.log(
    test_final["state_population"]
)


train_final = train_final.drop(
    columns=["state_population"]
)

test_final = test_final.drop(
    columns=["state_population"]
)

In [14]:
# save final train/test datasets

TRAIN_OUT = DATA_DIR / "team_rho_train_preprocessed_engineered_v2.csv"
TEST_OUT = DATA_DIR / "team_rho_test_preprocessed_engineered_v2.csv"


train_final.to_csv(
    TRAIN_OUT,
    index=False
)

test_final.to_csv(
    TEST_OUT,
    index=False
)


print("Saved:")
print(TRAIN_OUT)
print(TEST_OUT)

train_final.head()

Saved:
/Users/caseybrookshier/Team-Rho-Capstone-Project/Data/processed/Engineered_Data_v2/team_rho_train_preprocessed_engineered_v2.csv
/Users/caseybrookshier/Team-Rho-Capstone-Project/Data/processed/Engineered_Data_v2/team_rho_test_preprocessed_engineered_v2.csv


,State,Year,admission_rate_per_100k,admission_rate_lag2,neighbor_admission_rate_lag1,search_PC1,search_PC2,search_PC3,log_state_population
0,Alaska,2017,303.650691,297.929391,394.821026,-4.362284,-0.868199,-0.377299,13.515733
1,Alaska,2018,319.023002,289.533044,407.012291,-4.243584,-0.623294,-0.459835,13.509833
2,Alaska,2019,333.968100,303.650691,408.686371,-3.918116,-0.194698,-1.192840,13.505723
3,Alaska,2020,348.761827,319.023002,410.772841,-3.797099,-0.049901,-1.278749,13.502385
4,Arizona,2016,205.910436,223.530133,340.205703,0.978794,-0.049951,1.845417,15.753499


In [15]:
#done